<a href="https://colab.research.google.com/github/jmarrietar/LLMs-from-scratch-study/blob/feature%2Fstudy-session/Chapter_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Es necesario construir primero un diccionaro. El cual va a mapear cada una de las palabras y caracteres especiales a un numero unico. 

#### 2.2 Tokenizing text  

In [5]:
import urllib.request 
import re

url = ("https://raw.githubusercontent.com/rasbt/"
       "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
       "the-verdict.txt")

file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x7cdf931f6630>)

In [6]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("total number of characters:", len(raw_text))
print(raw_text[:99])

total number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


Como dividimos el texto para obtener la lista de tokens? 

In [7]:
# Una forma puede ser utilizando expresiones regulares re. Podemos usar 
# re.split para dividir en los caracteres en blanco 

text = "Hellow, world. This, is a test"

result = re.split(r"(\s)", text)

print(result)

['Hellow,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test']


podemos ver que las palabras aun estan conectadas a caracteres especiales de puntuacion. 

In [8]:
# Agregando caracteres especiales. 
result = re.split(r'([,.]|\s)', text)

# Eliminando los caracteres vacios. 
result = [item for item in result if item.strip()]

print(result)

['Hellow', ',', 'world', '.', 'This', ',', 'is', 'a', 'test']


In [9]:
# Otra forma de de tokenizarlo quitando question marks, quotation marks etc. 
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


Aqui ya tendriamos un tokenizador basico

In [10]:
# Ahora aplicarselo a todo el texto. 
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


Se tendrian entonces 4690 tokens en todo ese texto

In [11]:
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


#### 2.3 Converting tokens into token IDs

Este es el paso intermedio antes de convertirlo en embeddings

Ya todo el texto esta tokenizado en una variable llamada "preprocessed"

In [12]:
# Vamos a ordenar todos los tokens de forma alfabetica

all_words = sorted(set[str](preprocessed))    
vocab_size = len(all_words)
print(vocab_size)

1130


El la lista de tokenns unicos es de 1130 

con estos tokens unicos ya ordenados se puede crear el diccionario. 

In [13]:
vocab = {token:integer for integer, token in enumerate(all_words)}

for i, item in enumerate(vocab.items()):
    print(item)
    if i >=10:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)


¿Que pasa si ya tenemos los token_ids y queremos devolvernos a tener el texto?

Debemos crear un vocabulario inverso que mapee los ids a tokens de texto. 


In [14]:
class SimpleTokenizerV1: 
    def __init__(self, vocab):
        self.str_to_int = vocab # Toca vocabulario
        self.int_to_str = {i:s for s,i in vocab.items()} # creamos un vocab invertido

    def encode(self, text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text) # split de caracteres 

        # Ahora llevarlos a una lista 
        preprocessed = [item.strip() for item in preprocessed if item.strip()]

        ids = [self.str_to_int[s] for s in preprocessed]

        return ids

    def decode(self, ids): 
        text = " ".join([self.int_to_str[i] for i in ids])

        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text 

In [15]:
tokenizer = SimpleTokenizerV1(vocab) 

text = """It's the last he painted, you know", 
        Mrs. Gisburn said with a pardonable pride. """

ids = tokenizer.encode(text)
print(ids)

[56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 1, 5, 67, 7, 38, 851, 1108, 115, 754, 793, 7]


In [16]:
tokenizer.decode(ids)

'It\' s the last he painted, you know", Mrs. Gisburn said with a pardonable pride.'

Sucessfully converted back to the original text. 

#### 2.4 Adding special context tokens

Vamos a adicionar dos nuevos tokens <|unk|> (especialmente para tokens nuevos de palabras desconocidas que no hacen parte del training data) y <|endoftext|>. 

In [17]:
all_tokens = sorted(list(set(preprocessed)))

all_tokens.extend(["<|unk|>", "<|endoftext|>"])

In [18]:
# Actualizamos los numerales de los tokens 
vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [19]:
class SimpleTokenizerV2: 
    def __init__(self, vocab):
        self.str_to_int = vocab # Toca vocabulario
        self.int_to_str = {i:s for s,i in vocab.items()} # creamos un vocab invertido

    def encode(self, text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text) # split de caracteres 

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]

        # Ahora llevarlos a una lista 
        preprocessed = [item if item in self.str_to_int 
                        else "<|unk|>" for item in preprocessed]

        ids = [self.str_to_int[s] for s in preprocessed]

        return ids

    def decode(self, ids): 
        text = " ".join([self.int_to_str[i] for i in ids])

        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text 

In [20]:
text1 = "Hello, do you like tea? "
text2 = " Jose Miguel A. "

text = " <|endoftext|> ".join((text1, text2))
print(text)

Hello, do you like tea?  <|endoftext|>  Jose Miguel A. 


In [21]:
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text)) 

[1130, 5, 355, 1126, 628, 975, 10, 1131, 1130, 1130, 11, 7]


In [22]:
print(tokenizer.encode("Jose")) 

[1130]


Podemos ver varios 1130 y 1131 que serian los endoftext y las palabras desconocidas. 

In [23]:
# Para mirar lo que devuelve el decoder del tokenizador. 
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|>, do you like tea? <|endoftext|> <|unk|> <|unk|> A.


#### 2.5 Byte pair encoding

In [24]:
!pip install tiktoken --quiet

In [25]:
from importlib.metadata import version 

import tiktoken 
print("tiktoken version:", version("tiktoken")) 

tiktoken version: 0.12.0


In [26]:
tokenizer = tiktoken.get_encoding("gpt2")

In [27]:
 text = (
            "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
             "of someunknownPlace."
)

In [28]:
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [29]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


Como nota de color, el BPE tokenizer con el que fue entrenado GPT-2 y GPT-3 tiene un tamaño de vocabulario de 50257

#### 2.6 Data sampling with a sliding window

In [30]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [31]:
raw_text[0:10]

'I HAD alwa'

In [32]:
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


5145 tokens en total

In [33]:
enc_sample = enc_text[50:]

In [34]:
enc_sample

[290,
 4920,
 2241,
 287,
 257,
 4489,
 64,
 319,
 262,
 34686,
 41976,
 13,
 357,
 10915,
 314,
 2138,
 1807,
 340,
 561,
 423,
 587,
 10598,
 393,
 28537,
 2014,
 198,
 198,
 1,
 464,
 6001,
 286,
 465,
 13476,
 1,
 438,
 5562,
 373,
 644,
 262,
 1466,
 1444,
 340,
 13,
 314,
 460,
 3285,
 9074,
 13,
 46606,
 536,
 5469,
 438,
 14363,
 938,
 4842,
 1650,
 353,
 438,
 2934,
 489,
 3255,
 465,
 48422,
 540,
 450,
 67,
 3299,
 13,
 366,
 5189,
 1781,
 340,
 338,
 1016,
 284,
 3758,
 262,
 1988,
 286,
 616,
 4286,
 705,
 1014,
 510,
 26,
 475,
 314,
 836,
 470,
 892,
 286,
 326,
 11,
 1770,
 13,
 8759,
 2763,
 438,
 1169,
 2994,
 284,
 943,
 17034,
 318,
 477,
 314,
 892,
 286,
 526,
 383,
 1573,
 11,
 319,
 9074,
 13,
 536,
 5469,
 338,
 11914,
 11,
 33096,
 663,
 4808,
 3808,
 62,
 355,
 996,
 484,
 547,
 12548,
 287,
 281,
 13079,
 410,
 12523,
 286,
 22353,
 13,
 843,
 340,
 373,
 407,
 691,
 262,
 9074,
 13,
 536,
 48819,
 508,
 25722,
 276,
 13,
 11161,
 407,
 262,
 40123,
 18113,


x van a ser los input tokens y y los output tokens (con shift de 1)

In [35]:
context_size = 4 

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")


x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [36]:
# Si corremos los inputs al lado de cual seria el target seria algo como: 

for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "--->", desired)

[290] ---> 4920
[290, 4920] ---> 2241
[290, 4920, 2241] ---> 287
[290, 4920, 2241, 287] ---> 257


Checkpoint: Aqui por lo que estoy viendo, target es una palabra. 

In [37]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "--->", tokenizer.decode([desired]))


 and --->  established
 and established --->  himself
 and established himself --->  in
 and established himself in --->  a


Con esto ya estamos creando input, target pairs que nos sirven para entrenar LLMs. 

In [51]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset): 
    def __init__(self, txt, tokenizer, max_length, stride): 
        self.input_ids = []
        self.target_ids = []
    
        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i+1: i + max_length +1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [52]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=shuffle,
        drop_last=drop_last,
    )

    return dataloader

In [53]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [61]:
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)

In [62]:
data_iter = iter(dataloader)

first_batch = next(data_iter)

In [63]:
first_batch

[tensor([[15496,    11,   466,   345]]), tensor([[ 11, 466, 345, 588]])]

In [64]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 11, 466, 345, 588]]), tensor([[ 466,  345,  588, 8887]])]


Se puede hacer un stride mas grande por ejemplo de 4, esto podria ayudar a prevenir overlap. Y el overlap podria resultar en overfitting. 

#### 2.7 Creating token embeddings

In [65]:
torch.manual_seed(123)

In [66]:
# Digamos que tenemos de ejemplo:  

input_ids = torch.tensor([2, 3, 5, 1])

vocab_size = 6 
output_dim = 3

embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [69]:
embedding_layer.weight # Random values al inicio

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)

In [71]:
embedding_layer(torch.tensor([3]))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)

#### 2.8 Encoding word positions 